# 🤖 AI Agents under the Hood — Advanced

**Fortgeschrittene Themen, aufbauend auf dem Basics-Notebook** (`AI_Agents_under_the_Hood.ipynb`).

Das Basics-Notebook baut einen Agenten *from scratch*: LLM → Memory → Tools → Loop → ReAct → Coding → Context-Engineering (von Hand) → Harness → MCP. Dieses Notebook geht **tiefer** — derzeit:

| Thema | Frage | Zutat |
|---|---|---|
| **Context Management** | Wie verwaltet man das Kontextfenster *richtig* — was passiert unter der Haube? | `ctxman` (Service) |

> ⚠️ Voraussetzungen:
> 1. Der **ctxman-Service** läuft auf `http://localhost:5291` — Start: `./start-ctxman-azure-compaction.ps1` (im Ordner; braucht den laufenden Postgres-Container `ctxman-pg`).
> 2. Dieselbe **`.env`** wie das Basics-Notebook (Azure-OpenAI-Credentials).

In [ ]:
import os, json
from pathlib import Path
from openai import AzureOpenAI
from dotenv import load_dotenv

load_dotenv()  # liest die .env (gleiche wie Basics-Notebook)

client = AzureOpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION", "2024-10-21"),
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
)
DEPLOYMENT = os.environ["AZURE_OPENAI_DEPLOYMENT"]
print("Client bereit. Deployment:", DEPLOYMENT)

### Tool-Registry (aus dem Basics-Notebook)

Der Agentic Loop in Abschnitt 7 braucht eine kleine Tool-Registry. Wir übernehmen den Minimalsatz aus dem Basics-Notebook (Kap. 4), damit dieses Notebook eigenständig läuft.

In [ ]:
TOOLS, DISPATCH = [], {}

def register(name, description, parameters):
    def deco(fn):
        TOOLS.append({"type": "function", "function": {
            "name": name, "description": description, "parameters": parameters}})
        DISPATCH[name] = fn
        return fn
    return deco

@register("list_files", "Listet Dateien in einem Verzeichnis.",
          {"type": "object", "properties": {"path": {"type": "string"}}, "required": []})
def _list_files(path="."):
    return "\n".join(sorted(os.listdir(path)))

@register("read_file", "Liest den Textinhalt einer Datei (erste 2000 Zeichen).",
          {"type": "object", "properties": {"path": {"type": "string"}}, "required": ["path"]})
def _read_file(path):
    return Path(path).read_text(encoding="utf-8", errors="replace")[:2000]

print("Registrierte Tools:", list(DISPATCH))

## Context Management *unter der Haube* — `ctxman`

Im **Basics-Notebook** haben wir Context-Engineering **von Hand** gemacht: `count_tokens`, `truncate`, `compact`. Genau diese Ideen — nur als *echter, wiederverwendbarer Service* — stecken in **`ctxman`** (C#/.NET, REST-API). Statt sie in jedem Agenten neu zu erfinden, lagern wir die Frage **„Was sieht das Modell in genau diesem Turn?"** an einen spezialisierten Dienst aus.

**Mentales Modell — Context wie Speicherverwaltung:**

| Region | Analogie | Inhalt | Verhalten |
|---|---|---|---|
| **Static** | Stack | System-Prompt, Tool-Defs | cache-stabil, immutable je Epoche |
| **Working** | Heap | Conversation, Tool-Results | dynamisch; **GC** hält das Budget |

Was wir gleich **live** sehen:
- **Segmente** sind die Source of Truth — die `messages[]`-Liste ist nur ein **Render-Artefakt**. Der Agent hält selbst **keine** History mehr.
- **Minor GC** *externalisiert* Tool-Results (Wert → Pointer) — das ist `truncate`, aber verlustfrei (Page-Fault holt zurück).
- **Major GC** *kompaktiert* alte Turns per LLM (→ ein Summary-Segment) — das ist `compact`, serverseitig & auditierbar.
- **Events** machen jede Operation nachvollziehbar („Warum wusste der Agent das in Turn 30 nicht mehr?").

> ⚠️ Voraussetzung: ctxman läuft auf `http://localhost:5291` (Start siehe `start-ctxman-azure-compaction.ps1`). ctxman ruft **nie** das Modell des Agents — den LLM-Call macht weiter unser Notebook. Nur die *Compaction* nutzt ein separates, in ctxman konfiguriertes LLM-Backend.

In [25]:
# --- 7½ · Mini-Client: ctxman ist nur REST. Mehr braucht es nicht. ---
import urllib.request, uuid

CTX_BASE   = "http://localhost:5291"
CTX_TENANT = "talk-demo"

def _ctx(method, path, body=None):
    headers = {"Content-Type": "application/json",
               "X-Tenant-Id": CTX_TENANT,
               "Idempotency-Key": str(uuid.uuid4())}  # eindeutig: ein Retry zählt den Turn nicht doppelt
    data = json.dumps(body).encode() if body is not None else None
    with urllib.request.urlopen(urllib.request.Request(CTX_BASE + path, data=data,
                                                       headers=headers, method=method)) as r:
        return json.load(r)

def seg(**kw):
    "Voll besetztes Segment-DTO (ctxman verlangt alle Keys; nicht gesetzte = null)."
    base = {"kind": None, "role": None, "content": None, "blob_ref": None,
            "tool_call_id": None, "pinned": None, "source": None, "region": None}
    base.update(kw); return base

class Ctx:
    "Dünner ctxman-Client: append + render statt einer lokalen messages-Liste."
    def create(self, static_segments, policy=None):
        return _ctx("POST", "/v1/sessions",
                    {"agent_template_id": None, "policy_overrides": policy,
                     "static_segments": static_segments})["session_id"]
    def append(self, sid, *segments):
        return _ctx("POST", f"/v1/sessions/{sid}/segments", {**seg(), "segments": list(segments)})
    def render(self, sid, provider="openai", turn_advance=True):
        return _ctx("POST", f"/v1/sessions/{sid}/render",
                    {"provider": provider, "scope": "path", "turn_advance": turn_advance})
    def session(self, sid):    return _ctx("GET",  f"/v1/sessions/{sid}")
    def events(self, sid):     return _ctx("GET",  f"/v1/sessions/{sid}/events?after_seq=0")["events"]
    def gc(self, sid, level):  return _ctx("POST", f"/v1/sessions/{sid}/gc", {"level": level})
    def ref(self, sid, segid): return _ctx("GET",  f"/v1/sessions/{sid}/refs/{segid}")

ctx = Ctx()
print("ctxman:", _ctx("GET", "/healthz"))   # {'status': 'ok', ...} -> Service erreichbar

ctxman: {'status': 'ok', 'auth_mode': 'none'}


### 1 · Session anlegen — Static vs. Working

Der System-Prompt kommt in die **Static-Region** (cache-stabil). Danach hängen wir Working-Segmente an (`user_msg`, `assistant_msg`). Entscheidend: **wir** speichern die Konversation nicht mehr — ctxman besitzt sie als Segmente. Das Notebook hat keine `messages`-Liste.

In [26]:
sid = ctx.create(static_segments=[
    seg(kind="system_prompt", role="system", content="Du bist ein DevOps-Agent.", source="core"),
])
ctx.append(sid, seg(kind="user_msg",      role="user",      content="Welche Pods sind ungesund?"))
ctx.append(sid, seg(kind="assistant_msg", role="assistant", content="web-0 ist OOMKilled."))

s = ctx.session(sid)
print("session     :", sid)
print("current_turn:", s["current_turn"], "| tokens_used:", s["tokens_used"], "| watermark:", s["watermark_state"])
print("\n→ Im Notebook gibt es KEINE messages-Liste. Die Konversation lebt als Segmente in ctxman.")

session     : 01KV09FN4SYMCG8CZDX4F7W1H7
current_turn: 0 | tokens_used: 19 | watermark: ok

→ Im Notebook gibt es KEINE messages-Liste. Die Konversation lebt als Segmente in ctxman.


### 2 · `render` — dasselbe Segment-Set, zwei Provider-Formate

`render` ist der Kern: aus den Segmenten erzeugt ein **Provider-Adapter** das fertige Request-Fragment. Dieselben Segmente → **Anthropic** *oder* **OpenAI** Wire-Format. Dazu liefert ctxman `builtin_tools` (das Page-Fault-Tool `expand_context_ref`), `cache_breakpoints` (für Prompt-Caching) und den `watermark_state`.

In [27]:
a = ctx.render(sid, provider="anthropic", turn_advance=False)
o = ctx.render(sid, provider="openai",    turn_advance=False)

print("— ANTHROPIC —  (system ist Top-Level-Feld)")
print("  system  :", a["request_fragment"]["system"])
print("  messages:", json.dumps(a["request_fragment"]["messages"])[:150])
print("\n— OPENAI —  (system ist die erste Message)")
for m in o["request_fragment"]["messages"]:
    print("   ", m["role"], "=>", str(m["content"])[:60])

print("\nbuiltin_tools  :", a["builtin_tools"][0]["name"], "(Page-Fault-Tool)")
print("cache_breakpts :", a["cache_breakpoints"])
print("watermark/tok  :", a["watermark_state"], "/", a["tokens_total"], "Tokens")

— ANTHROPIC —  (system ist Top-Level-Feld)
  system  : Du bist ein DevOps-Agent.
  messages: [{"role": "user", "content": [{"type": "text", "text": "Welche Pods sind ungesund?"}]}, {"role": "assistant", "content": [{"type": "text", "text": "we

— OPENAI —  (system ist die erste Message)
    system => Du bist ein DevOps-Agent.
    user => Welche Pods sind ungesund?
    assistant => web-0 ist OOMKilled.

builtin_tools  : expand_context_ref (Page-Fault-Tool)
cache_breakpts : [{'kind': 'static_region_end', 'index': 0}]
watermark/tok  : ok / 19 Tokens


### 3 · Minor GC — Externalisierung (Wert → Pointer)

Ein großes `tool_result` (≈ 50 000 Zeichen) flutet den Context. **Minor GC** schreibt den Inhalt in einen Blob-Store und ersetzt ihn im Render durch **Summary + Pointer** (`expand_context_ref(...)`); der zugehörige `tool_call` bleibt intakt (vollständige Unit). Das ist `truncate` aus Kapitel 7 — aber **verlustfrei**.

> `tokens_total` zählt ctxman beim *Append* (konservativ) und rechnet beim Render nicht neu — die echte Schrumpfung sehen wir an der **gerenderten Länge** des Tool-Results.

In [28]:
import time

xs = ctx.create(static_segments=[seg(kind="system_prompt", role="system", content="DevOps-Agent.", source="core")])
ctx.append(xs, seg(kind="tool_call", role="assistant", content="kubectl_logs()", tool_call_id="c1", source="kubectl_logs"))
big = "CrashLoopBackOff web-0 restart=7; OOMKilled; " * 1100          # ~ 50 000 Zeichen
ext_id = ctx.append(xs, seg(kind="tool_result", role="tool", content=big, tool_call_id="c1"))["segment_ids"][0]

def tool_result_view(sid):
    msgs = ctx.render(sid, "anthropic", turn_advance=False)["request_fragment"]["messages"]
    blk = [b for m in msgs for b in m["content"] if b.get("type") == "tool_result"][0]
    return len(blk["content"]), ("expand_context_ref" in blk["content"])

n0, ptr0 = tool_result_view(xs)
print(f"VOR  Minor GC: tool_result = {n0:>6} Zeichen | ist Pointer: {ptr0}")

ctx.gc(xs, "minor")                                                   # async im Service
for _ in range(15):
    time.sleep(1)
    if any(e["type"] == "segment_externalized" for e in ctx.events(xs)): break

n1, ptr1 = tool_result_view(xs)
print(f"NACH Minor GC: tool_result = {n1:>6} Zeichen | ist Pointer: {ptr1}   ← Wert wurde ausgelagert")
print("Events:", [e["type"] for e in ctx.events(xs)])

VOR  Minor GC: tool_result =  49500 Zeichen | ist Pointer: False
NACH Minor GC: tool_result =    286 Zeichen | ist Pointer: True   ← Wert wurde ausgelagert
Events: ['segment_appended', 'render_served', 'segment_externalized', 'render_served']


### 4 · Page-Fault — `expand_context_ref`

Der Pointer ist kein Datenverlust: ruft das Modell `expand_context_ref(segment_id)` auf, holt das SDK den Volltext per `GET /refs/{id}` zurück (und setzt `last_referenced_turn` → approximierte Liveness). Genau das passiert beim „Nachschlagen" eines ausgelagerten Ergebnisses.

In [29]:
full = ctx.ref(xs, ext_id)
print("Page-Fault GET /refs liefert", len(full["content"]), "Zeichen aus dem Blob-Store zurück.")
print("Anfang:", full["content"][:70], "…")

Page-Fault GET /refs liefert 49500 Zeichen aus dem Blob-Store zurück.
Anfang: CrashLoopBackOff web-0 restart=7; OOMKilled; CrashLoopBackOff web-0 re …


### 5 · Major GC — Compaction & Promotion (LLM-gestützt)

Stapeln sich viele alte Turns, fasst **Major GC** sie per LLM-Call zu **einem** `compaction_summary`-Segment zusammen — das ist `compact` aus Kapitel 7, aber serverseitig mit einem *separaten* Modell und **asynchron**. Davor läuft zwingend **Promotion**: dauerhafte Fakten (Entscheidungen, Constraints) werden an einen **Memory-Sink** (Webhook) geschrieben, *bevor* komprimiert wird (Compaction ist lossy).

Wir stellen den Memory-Sink als **winzigen lokalen Webhook** im Notebook nach — so sehen wir den **extrahierten Fakt** live ankommen.

In [17]:
import time, threading
from http.server import BaseHTTPRequestHandler, HTTPServer

promoted = []   # was unser lokaler "Memory-Store" empfängt
class _Sink(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"
    def do_POST(self):
        cl, te = self.headers.get("Content-Length"), (self.headers.get("Transfer-Encoding") or "")
        if cl is not None:                                   # normaler Body
            raw = self.rfile.read(int(cl))
        elif "chunked" in te.lower():                        # .NET streamt JSON chunked (kein Content-Length)
            raw = b""
            while (size := int(self.rfile.readline().strip() or b"0", 16)):
                raw += self.rfile.read(size); self.rfile.readline()
            self.rfile.readline()
        else:
            raw = b""
        try: promoted.append(json.loads(raw))
        except Exception: pass
        self.send_response(200); self.send_header("Content-Length", "2"); self.end_headers(); self.wfile.write(b"{}")
    def log_message(self, *a): pass
_srv = HTTPServer(("127.0.0.1", 0), _Sink); _sink_port = _srv.server_address[1]
threading.Thread(target=_srv.serve_forever, daemon=True).start()
print("lokaler Memory-Sink:", f"http://localhost:{_sink_port}/ingest")

policy = {
    "budget_tokens": 4000, "watermarks": {"soft": 0.6, "hard": 0.8, "emergency": 0.95},
    "externalize_threshold_tokens": 2000, "tokenizer": None, "on_tool_removed": None,
    "compaction": {"model": "gpt-5.4-mini", "prompt_template_id": "default-v1", "max_share": 0.8},
    "promotion": {"sink": {"type": "webhook", "url": f"http://localhost:{_sink_port}/ingest"}},
}
cs = ctx.create(static_segments=[seg(kind="system_prompt", role="system", content="DevOps-Agent.", source="core")], policy=policy)
for i in range(4):
    ctx.append(cs, seg(kind="user_msg", role="user", content=f"Runde {i}: Was ist mit Pod web-{i}?"))
    ctx.append(cs, seg(kind="assistant_msg", role="assistant",
        content=(f"Untersuchung web-{i} (namespace prod): wiederholt OOMKilled (exit 137). Ursache: Memory-Limit "
                 f"128Mi zu niedrig fuer den JVM-Heap. ENTSCHEIDUNG: Limit dauerhaft auf 512Mi angehoben, "
                 f"HPA-Schwelle auf 70% CPU. Ticket OPS-{1000+i} geschlossen.")))

tok_before = ctx.session(cs)["tokens_used"]
ctx.render(cs, "openai", turn_advance=True)
print("VOR  Major GC: tokens_used =", tok_before)

# Major GC ist async + nutzt ein echtes LLM-Backend -> kleiner Retry gegen transiente 5xx
done = False
for attempt in range(4):
    ctx.gc(cs, "major")
    for _ in range(30):
        time.sleep(1)
        evs = ctx.events(cs)
        if any(e["type"] == "compaction_completed" for e in evs): done = True; break
    if done: break
    time.sleep(4)

if not done:
    print("Major GC nicht abgeschlossen — das Compaction-Backend lieferte transiente 5xx. Zelle einfach erneut ausführen.")
else:
    cc = [e for e in evs if e["type"] == "compaction_completed"][0]["payload"]
    print(f"compaction_completed: {len(cc['source_ids'])} Segmente → 1 Summary "
          f"(tokens {cc['tokens_before']} → {cc['tokens_after']})")
    print("NACH Major GC: tokens_used =", ctx.session(cs)["tokens_used"], " ← alte Turns sind state=compacted")
    fact = promoted[-1].get("fact") if promoted else None
    print("\nPromoted Fact (extrahiert & an den Memory-Sink geschickt, VOR der Compaction):")
    print(" ", (fact or "(Promotion lieferte keinen dauerhaften Fakt)")[:400])
_srv.shutdown()

lokaler Memory-Sink: http://localhost:56872/ingest
VOR  Major GC: tokens_used = 272
compaction_completed: 8 Segmente → 1 Summary (tokens 268 → 83)
NACH Major GC: tokens_used = 87  ← alte Turns sind state=compacted

Promoted Fact (extrahiert & an den Memory-Sink geschickt, VOR der Compaction):
  - Betroffene Pods: `web-0`, `web-1`, `web-2`, `web-3` im Namespace `prod`
- Symptom bei allen Pods: wiederholt `OOMKilled` mit Exit-Code `137`
- Ursache: das Memory-Limit von `128Mi` war zu niedrig für den JVM-Heap
- Entscheidung: Memory-Limit dauerhaft auf `512Mi` erhöht
- Weitere Maßnahme: HPA-Schwelle auf `70%` CPU gesetzt
- Status der Tickets: `OPS-1000`, `OPS-1001`, `OPS-1002`, `OPS-1003` wur


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 56905)
Traceback (most recent call last):
  File "C:\Users\rudi\AppData\Roaming\uv\python\cpython-3.13.1-windows-x86_64-none\Lib\socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
    ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\rudi\AppData\Roaming\uv\python\cpython-3.13.1-windows-x86_64-none\Lib\socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\rudi\AppData\Roaming\uv\python\cpython-3.13.1-windows-x86_64-none\Lib\socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
    ~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\rudi\AppData\Roaming\uv\python\cpython-3.13.1-windows-x86_64-none\Lib\socketserver.py", line 761, in __ini

### 6 · Event-Stream — das Audit *unter der Haube*

Jede Mutation und jede GC-Operation emittiert ein **Event** (Outbox-Pattern). Das ist die Antwort auf *„Warum wusste der Agent das in Turn N nicht mehr?"* — man liest es einfach im Stream nach. Hier die Timeline der Externalisierungs-Session aus Abschnitt 3:

In [ ]:
for e in ctx.events(xs):
    print(f"  seq {e['seq']:>2}  {e['type']:<22} {json.dumps(e['payload'])[:80]}")

### 7 · Der Agentic Loop *auf* ctxman

Jetzt der Lohn: derselbe Loop wie im Basics-Notebook — aber er hält **keine `messages`-Liste** mehr. Pro Turn: `append` → `render` → Modell-Call → Tool ausführen → `append`. Der Context lebt komplett im Service; Tool-Defs liegen in der Static-Region, die Konversation in Working-Segmenten. (Tool-Registry `TOOLS`/`DISPATCH` ist oben in diesem Notebook definiert.)

In [18]:
def run_agent_ctx(task, max_steps=8):
    "Agentic Loop OHNE lokale messages-Liste — der Context lebt in ctxman."
    statics = [seg(kind="system_prompt", role="system",
                   content="Du bist ein hilfreicher Agent. Nutze Tools, wenn nötig.", source="core")]
    for t in TOOLS:                                    # Tool-Defs in die Static-Region
        statics.append(seg(kind="tool_def", content=json.dumps(t["function"]), source="core"))
    sid = ctx.create(static_segments=statics)
    ctx.append(sid, seg(kind="user_msg", role="user", content=task))

    for step in range(1, max_steps + 1):
        frag = ctx.render(sid, provider="openai", turn_advance=True)["request_fragment"]   # 1 Turn = 1 Modell-Call
        msg = client.chat.completions.create(
            model=DEPLOYMENT, messages=frag["messages"], tools=frag["tools"]).choices[0].message
        if not msg.tool_calls:
            print(f"[schritt {step}] ✓ finale Antwort")
            return msg.content, sid
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments or "{}")
            print(f"[schritt {step}] → {tc.function.name}({args})")
            ctx.append(sid, seg(kind="tool_call", role="assistant",
                                content=tc.function.arguments, tool_call_id=tc.id, source=tc.function.name))
            try:    result = str(DISPATCH[tc.function.name](**args))
            except Exception as e: result = f"ERROR: {e}"
            ctx.append(sid, seg(kind="tool_result", role="tool", content=result, tool_call_id=tc.id))
    return "(max_steps erreicht)", sid

answer, sid_loop = run_agent_ctx("Wie viele Dateien liegen im aktuellen Verzeichnis? Nenne nur die Zahl und die Namen.")
print("\n", answer)
print("\n→ Das Notebook führte KEINE messages-Liste. ctxman besitzt die History:",
      ctx.session(sid_loop)["current_turn"], "Turns,", ctx.session(sid_loop)["tokens_used"], "Tokens.")

URLError: <urlopen error [WinError 10061] No connection could be made because the target machine actively refused it>

### Wrap-up — `ctxman` ist Kapitel 7, aber als Service

| Handarbeit (Basics-Notebook) | ctxman *unter der Haube* |
|---|---|
| `messages`-Liste im Notebook | **Segmente** im Service (Source of Truth) |
| `truncate(text)` | **Externalisierung** (Wert → Pointer, verlustfrei) |
| — | **Page-Fault** `expand_context_ref` (lazy zurückholen) |
| `compact(messages)` | **Major GC**: Compaction + Promotion (LLM, async) |
| `count_tokens` / Budget im Loop | **Watermarks** + automatischer GC |
| `print()`-Logging | **Event-Stream** (auditierbar) |

**Kernidee:** Context-Management ist eine eigene, wiederverwendbare Disziplin — Speicherverwaltung für das Kontextfenster. Der Agent fragt nur noch *„render"* und bekommt das fertige Fragment; *was* drinsteht, entscheidet der Service nach deklarativer Policy.